# 第5回：性能を向上させるテクニック

比較条件を固定し、特徴量、モデル設定、しきい値を1つずつ改善します。

上から順に実行してください。


In [ ]:
from pathlib import Path

def find_repo_root(start=None):
    current = Path.cwd() if start is None else Path(start)
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある教材フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## 外部資料と組み合わせる

このNotebookは第5回の最低限の実習です。
実習の前後に[外部資料の指定範囲](../../docs/resources.md)を読み、確認問題と追加演習にも取り組みます。


## 改善のルール

1. 比較条件を固定する
2. 変更は1つにする
3. 良化も悪化も記録する
4. 最後に未使用データで確認する


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_predict,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import make_pipeline

data = pd.read_csv(DATA / "compound_experiments.csv")
base_features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
cv = StratifiedKFold(5, shuffle=True, random_state=42)
development, final_test = train_test_split(
    data, test_size=0.2, random_state=42, stratify=data["active"]
)
print("改善に使う件数:", len(development), "最終確認用:", len(final_test))


## 基準となるモデル

最終確認用データには触れず、改善用データを同じ5分割交差検証で比べます。


In [ ]:
base_model = make_pipeline(
    SimpleImputer(strategy="median"),
    HistGradientBoostingClassifier(max_iter=150, random_state=42),
)
base_scores = cross_val_score(
    base_model, development[base_features], development["active"], cv=cv, scoring="f1"
)
print(f"基準F1: {base_scores.mean():.3f} ± {base_scores.std():.3f}")


## 1. 知識から特徴量を作る

温度78℃からの距離と、濃度×反応時間を追加します。元の列は残したまま効果を比べます。


In [ ]:
improved = development.copy()
improved["temperature_distance"] = (improved["temperature_c"] - 78).abs()
improved["concentration_time"] = improved["concentration_m"] * improved["reaction_time_h"]
improved_features = [*base_features, "temperature_distance", "concentration_time"]

feature_scores = cross_val_score(
    base_model, improved[improved_features], improved["active"], cv=cv, scoring="f1"
)
print(f"特徴量追加後: {feature_scores.mean():.3f} ± {feature_scores.std():.3f}")


## 2. モデル設定を探索する

候補を限定し、交差検証の内側で良い設定を探します。


In [ ]:
search = RandomizedSearchCV(
    base_model,
    param_distributions={
        "histgradientboostingclassifier__learning_rate": [0.03, 0.05, 0.08, 0.12],
        "histgradientboostingclassifier__max_leaf_nodes": [7, 15, 31, 63],
        "histgradientboostingclassifier__l2_regularization": [0.0, 0.1, 1.0],
    },
    n_iter=8,
    scoring="f1",
    cv=cv,
    random_state=42,
)
search.fit(improved[improved_features], improved["active"])
print("探索後F1:", round(search.best_score_, 3))
print("設定:", search.best_params_)


## 3. しきい値を調整する

各行が検証側になったときの確率を集めます。この予測をOOF予測と呼び、しきい値選びに使います。


In [ ]:
oof_probability = cross_val_predict(
    search.best_estimator_, improved[improved_features], improved["active"],
    cv=cv, method="predict_proba"
)[:, 1]
thresholds = np.arange(0.20, 0.81, 0.02)
threshold_scores = [f1_score(improved["active"], oof_probability >= value) for value in thresholds]
best_threshold = thresholds[int(np.argmax(threshold_scores))]
print(f"最良しきい値={best_threshold:.2f}, OOF F1={max(threshold_scores):.3f}")


## 4. 重要な列を確認する

改善用データをもう一度分け、列を1つずつ並べ替えたときにF1がどれだけ下がるかを測ります。


In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    improved[improved_features], improved["active"], test_size=0.25,
    random_state=7, stratify=improved["active"]
)
final_model = search.best_estimator_.fit(X_train, y_train)
importance = permutation_importance(
    final_model, X_valid, y_valid, scoring="f1", n_repeats=10, random_state=42
)
pd.DataFrame({"特徴量": improved_features, "重要度": importance.importances_mean}).sort_values(
    "重要度", ascending=False
).round(3)


## 5. 最後に1回だけ確認する

選んだ特徴量・設定・しきい値を固定し、取り分けておいたデータでF1を確認します。


In [ ]:
final_data = final_test.copy()
final_data["temperature_distance"] = (final_data["temperature_c"] - 78).abs()
final_data["concentration_time"] = final_data["concentration_m"] * final_data["reaction_time_h"]

final_model = search.best_estimator_.fit(improved[improved_features], improved["active"])
final_probability = final_model.predict_proba(final_data[improved_features])[:, 1]
final_prediction = final_probability >= best_threshold
print("最終確認のF1:", round(f1_score(final_data["active"], final_prediction), 3))


## 演習

特徴量、モデル設定、しきい値のうち1つだけ変更し、変更前後の平均F1と標準偏差を記録してください。


## 振り返り

1. 同時に複数条件を変えない理由は何か
2. 探索も交差検証の内側で行う理由は何か
3. 並べ替え重要度が答える問いは何か
